# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayyanatariq03-arch/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ayyanatariq03-arch/Flyrank-ML"
REPO_DIR = "Flyrank-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content':     f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_mar':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}
print("Setup done. Ready to query.")

Setup done. Ready to query.


## 1. Unit of analysis + time window

**One row** = one content page, on one day, for one client
(client_hash_id + content_hash_id + report_date), for my Refresh/Content
Opportunity Scoring lane.

**Time window:** a mid-panel month, `month=2026-03`, from
`fact_content_daily_performance`. I avoid the sealed final month (June 2026 /
the `_sample` table), since that's reserved as the future outcome window for
later modeling weeks.

**Table(s) used:** `fact_content_daily_performance` (month=2026-03) joined to
`dim_content` on `content_hash_id`.

**What I'd predict/rank (proxy):** whether a page's impressions declined
month-over-month — a proxy for "needs review."

**Deliberately excluded:** any rebuilt product decision flag (e.g.
`health_score`, `priority_score`) — using one as a feature would just teach
a model to copy an existing rule instead of finding real signal.

Verified with real queries in Section 3.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Features (max 5):**
1. `gsc_impressions` — observed exposure, known at report_date
2. `gsc_clicks` — observed engagement, known at report_date
3. `gsc_avg_position` — observed ranking, known at report_date
4. `content_age_days` (dim_content) — known before any decision point
5. `word_count` (dim_content) — known since publish, well before decision point

**Label/proxy:** month-over-month impression decline, derived by comparing
early vs late impressions within the window (built in Section 3).

**Context (join keys, not features):** `client_hash_id`, `content_hash_id` —
used only for joins and grouping, never fed to a model.

**Excluded on purpose:** any product decision flag (`health_score`,
`priority_score`, `action_type`). Not shipped in this release, and even if
they were, using them as a feature would leak the answer — the model would
just learn to copy an existing rule instead of discovering its own signal.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)


In [4]:
grain_check = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id || content_hash_id || report_date) AS unique_grain_combos
    FROM {TABLES['fact_daily_mar']}
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date  unique_grain_combos
0    9841378 2026-03-01 2026-03-31              9841378


In [5]:
avail_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) AS rows_with_impressions
    FROM {TABLES['fact_daily_mar']}
""").df()
print(avail_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_impressions
0     9841378                9841378


In [12]:
schema_check = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
print(schema_check.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [11]:
features = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions) AS impressions_month,
           SUM(f.gsc_clicks) AS clicks_month,
           AVG(f.gsc_avg_position) AS avg_position,
           ANY_VALUE(DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')) AS content_age_days,
           ANY_VALUE(d.word_count) AS word_count
    FROM {TABLES['fact_daily_mar']} f
    JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    GROUP BY 1, 2
    HAVING impressions_month > 0
""").df()

print(f"{len(features):,} content items")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 content items


,client_hash_id,content_hash_id,impressions_month,clicks_month,avg_position,content_age_days,word_count
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,396,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,396,<NA>
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,396,<NA>
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,396,<NA>
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,396,2475


## Feature "Available When?" Lines

1. `impressions_month` — knowable at the decision moment because it's the
   sum of the prior month's observed traffic, already recorded by report_date.
2. `clicks_month` — same: fully observed by end of the feature window.
3. `avg_position` — observed ranking data, available as soon as GSC reports it.
4. `content_age_days` — known the moment the page was published, long before
   any decision point.
5. `word_count` — fixed at publish time, known well before the decision.

In [13]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

features["is_declining"] = (features["impressions_month"] < features["impressions_month"].median()).astype(int)

X_honest = features[["clicks_month", "avg_position", "content_age_days", "word_count"]].fillna(0)
y = features["is_declining"]

honest_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
print(f"Honest score (no leak): {honest_tree.score(X_honest, y):.3f}")

# THE TRAP: add impressions_month itself (label-derived) as a feature
X_leaky = features[["clicks_month", "avg_position", "content_age_days", "word_count", "impressions_month"]].fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
print(f"Leaky score (with impressions_month): {leaky_tree.score(X_leaky, y):.3f}  <- jumps toward perfect")

print(f"\nKept for real use: honest score = {honest_tree.score(X_honest, y):.3f}")

Honest score (no leak): 0.799
Leaky score (with impressions_month): 1.000  <- jumps toward perfect

Kept for real use: honest score = 0.799


## The Trap — Deliberate Leakage

Adding `impressions_month` as a feature — the exact column the label was
derived from — pushed the score toward perfect (1.000). That's leakage: the
"feature" is the answer in disguise. I removed it and kept the honest score
(0.799), which reflects real, usable signal built only from features
knowable before the decision point.

## 4. Data limits

- **Unbalanced panel:** only 9 of 70 clients have 12+ months of history —
  seasonality analysis is unreliable for most clients.
- **GSC-only early rows:** rows before a client's `ga4_data_start` have
  `ga4_data_available = FALSE` — engagement features are missing, not zero,
  for that period. Must check `dim_clients` before assuming "no traffic."
- **Window overlap risk:** if a feature window and target window touch the
  same days, any decline "prediction" is really just describing the present,
  not forecasting the future — this must be checked every time a window is
  redefined.
- **Sparse AI-session data:** only ~30K rows out of ~79M have AI session
  data, so this slice can't support AI-referral claims.

In [14]:
print("Limitation: unbalanced panel, GSC-only early history, window-overlap risk")

Limitation: unbalanced panel, GSC-only early history, window-overlap risk


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.